In [ ]:
import numpy as np
import hashlib

# Problem 1: Representing SHA-256 Data in Python

This notebook explains the choices I made for representing SHA-256 data, with a small demo for each one.

## Background

SHA-256 is defined in terms of 32-bit words, 512-bit message blocks and big-endian byte order. Python's built-in `int` has arbitrary precision, so it never overflows and has no fixed size. That doesn't match the standard on its own, so I looked at how to represent each piece of data. Because Problems 2 and 3 require NumPy, I use NumPy's fixed-width `uint32` type for words from the start, so the whole project uses one consistent convention.

## Summary

| Concept | Representation |
|---|---|
| 32-bit word | `np.uint32` |
| Sequence of 32-bit words | NumPy array with `dtype=np.uint32` |
| Input message | `bytes` |
| 512-bit message block | 64 bytes, read as an array of 16 `uint32` words |
| Final 256-bit hash | 8 `uint32` words, then 32 `bytes`, then 64 hex characters |
| Big-endian integers | `int.from_bytes(..., "big")` and the `">u4"` dtype |

In [ ]:
# 1. 32-bit words

# a normal python int just keeps growing
big = 0xFFFFFFFF + 1
print("python int:", hex(big))

# a uint32 wraps back to 0
x = np.array([0xFFFFFFFF], dtype=np.uint32)
y = np.array([1], dtype=np.uint32)
total = x + y
print("uint32:", hex(int(total[0])))

# rotate right (used a lot in sha-256)
def rotr(value, n):
    value = np.uint32(value)
    left = np.uint32(32 - n)
    n = np.uint32(n)
    return (value >> n) | (value << left)

print(hex(int(rotr(0x00000001, 1))))
print(hex(int(rotr(0x12345678, 8))))

python int: 0x100000000
uint32: 0x0
0x80000000
0x78123456


## 2. Sequences of 32-bit words

**Choice:** a NumPy array with `dtype=np.uint32`.

SHA-256 uses several fixed-length sequences of words: the 16-word message block, the 64-word message schedule, the 8 hash values, the 8 working variables and the 64 round constants. They are all indexed by position, so an array fits well, and every element keeps the 32-bit wraparound behaviour.

The `dtype` must always be written explicitly. Without it NumPy picks a default (usually 64-bit) and the 32-bit guarantee is lost. The code cell below builds the initial hash values and an empty 64-word schedule and checks their type and length.

In [5]:
# 2. sequences of 32-bit words

H = np.array([0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
              0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19], dtype=np.uint32)

W = np.zeros(64, dtype=np.uint32)

print("H:", H.dtype, H.shape)
print("W:", W.dtype, W.shape)

# without dtype numpy picks 64 bit
test = np.array([1, 2, 3])
print("no dtype:", test.dtype)

H: uint32 (8,)
W: uint32 (64,)
no dtype: int64


## 3. Input messages

**Choice:** `bytes`.

The standard defines the message as a string of bits, but real data (files, network data, encoded text) arrives as whole bytes. `bytes` is immutable, indexable and sliceable. Text has to be encoded before hashing, and the encoding is part of the input: the letter "é" is two bytes in UTF-8 and one byte in Latin-1, so the hashes would differ. The message length in bits is `8 * len(msg)`, which is needed later for padding. The code cell below shows this.

In [3]:
# 3. input messages

msg = b"abc"
print(msg)
print("bytes:", len(msg))
print("bits:", 8 * len(msg))

# text has to be encoded first
text = "é"
utf8 = text.encode("utf-8")
latin1 = text.encode("latin-1")
print(utf8)
print(latin1)
print(hashlib.sha256(utf8).hexdigest())
print(hashlib.sha256(latin1).hexdigest())

b'abc'
bytes: 3
bits: 24
b'\xc3\xa9'
b'\xe9'
4a99557e4033c3539de2eb65472017cad5f9557f7a0625a09f1c3f6e2ba69c4c
de2e331d891ae267a7009cb45b4e8830f170e0c937288ea2731a1941c7a53b0d


## 4. 512-bit message blocks

**Choice:** 64 bytes of the padded message, read as an array of 16 `uint32` words.

512 bits is 64 bytes, which is 16 words of 4 bytes each. Padding adds a single 1 bit (the byte `0x80`), then zero bytes until the length is 56 mod 64, then the original length in bits as a 64-bit big-endian number. The result is a multiple of 64 bytes, so it can be cut into blocks by slicing. Each block is converted to words with `np.frombuffer(block, dtype=">u4")`, which reads the bytes as big-endian, followed by `.astype(np.uint32)` to get ordinary native `uint32` values for calculation.

For the message "abc" this gives exactly one block. The first word is `0x61626380` (the bytes for a, b, c and the 1 bit) and the last word is 24 (the length in bits). A message of 56 bytes or more has no room for the length in its first block, so it needs a second block. Problem 4 handles that properly.

In [3]:
# 4. 512-bit message blocks

def pad(msg):
    bit_length = 8 * len(msg)
    padded = msg + b"\x80"
    while len(padded) % 64 != 56:
        padded = padded + b"\x00"
    padded = padded + bit_length.to_bytes(8, "big")
    return padded

def split_blocks(padded):
    blocks = []
    for i in range(0, len(padded), 64):
        blocks.append(padded[i:i + 64])
    return blocks

def block_to_words(block):
    words = np.frombuffer(block, dtype=">u4")
    return words.astype(np.uint32)

padded = pad(b"abc")
blocks = split_blocks(padded)
words = block_to_words(blocks[0])

print("padded length:", len(padded))
print("number of blocks:", len(blocks))
print("words:", words.dtype, words.shape)
print("first word:", hex(int(words[0])))
print("last word:", int(words[15]))

# 56 bytes does not leave room for the length, so 2 blocks
long_blocks = split_blocks(pad(b"a" * 56))
print("56 byte message blocks:", len(long_blocks))

padded length: 64
number of blocks: 1
words: uint32 (16,)
first word: 0x61626380
last word: 24
56 byte message blocks: 2


## 5. The final 256-bit hash value

**Choice:** 8 `uint32` words during the calculation, and at the end 32 `bytes` shown as 64 hex characters.

The algorithm finishes with the eight words H0 to H7. Writing each one big-endian and joining them gives the 256-bit digest. Bytes are the raw form, and hex is the form people read and compare. The byte order matters: converting with `astype(">u4")` before `tobytes()` guarantees big-endian output. Calling `tobytes()` directly would use the machine's native order, which is little-endian on most computers, and give the wrong digest. The code cell below rebuilds the known SHA-256 of "abc" from its eight words and confirms it matches `hashlib`. `hashlib` is part of Python's standard library and is only used to check results.

In [4]:
# 5. final 256-bit hash value

# the final 8 words for "abc"
final_words = np.array([0xba7816bf, 0x8f01cfea, 0x414140de, 0x5dae2223,
                        0xb00361a3, 0x96177a9c, 0xb410ff61, 0xf20015ad],
                       dtype=np.uint32)

digest = final_words.astype(">u4").tobytes()
hex_str = digest.hex()

print("bytes:", len(digest))
print("hex characters:", len(hex_str))
print(hex_str)

# check against hashlib (only used to check)
print("matches hashlib:", hex_str == hashlib.sha256(b"abc").hexdigest())

# wrong way: tobytes() without big-endian
wrong_hex = final_words.tobytes().hex()
print("wrong order matches:", wrong_hex == hex_str)

bytes: 32
hex characters: 64
ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
matches hashlib: True
wrong order matches: False


### Problem 2: SHA-256 Bitwise Operations

### Problem 3: Generating the SHA-256 Constants

### Problem 4: Padding and Parsing Messages

### Problem 5: The SHA-256 Compression Function

### Problem 6: Complete SHA-256